# Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect  signatures of anomalous or strongly anomalous diffusion.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
import numpy as np
import matplotlib.pyplot as plt
from src import (
    integrate_to_velocity,
    integrate_to_displacement,
    compute_moment_scaling,
    compute_increments,
    compute_moments_from_increments,
    compute_scaling_exponents,
    test_scaling_linearity,
    fit_piecewise_scaling,
    trim_to_event_window,
    build_scaling_summary,
    set_plot_style,
    plot_onset_diagnostic,
    plot_onset_distribution,
    plot_increments_histograms_dual_view,
    identify_windows_all_files,
    compute_and_save_all_windowed
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Data loading

In [ ]:
logger.info("Loading preprocessed data...")
df_acc = pd.read_parquet('../data/processed/02_signals/acc_preprocessed_scaling.parquet')
check(len(df_acc) > 0, f"All signals loaded: {df_acc['file'].nunique()} files")

# Create output directories
DATA_DIR = Path('../data/processed/04b_moment_scaling')
DATA_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = Path('../figures/04_spatial/04b_moment_scaling')
for process in ['acceleration', 'velocity', 'displacement']:
    (FIGURES_DIR / process).mkdir(parents=True, exist_ok=True)

## 3. Temporal window detection

Detect four temporal windows in each signal using STA/LTA (Short-Term Average / Long-Term Average) method:

- **Pre-arrival**: Before P-wave onset (instrumental noise)
- **P-wave**: From P-onset to PGA (primary waves)
- **S-wave**: From PGA to ~100s after (secondary waves, main event)
- **Coda**: Decay phase after S-wave

Each station has its own window boundaries based on actual wave arrival times.

In [ ]:
logger.info("Detecting temporal windows for all files...")

# Check if windows already exist
windows_path = DATA_DIR / 'temporal_windows.parquet'

if windows_path.exists():
    logger.info("Loading existing window detection results...")
    df_windows = pd.read_parquet(windows_path)
    logger.info(f"Windows loaded: {len(df_windows)} windows for {df_windows['file'].nunique()} files")
else:
    logger.info("Running window detection (this may take a few minutes)...")
    df_windows = identify_windows_all_files(
        df_acc,
        sampling_rate=200,
        sta_lta_threshold=3.0,
        output_dir=FIGURES_DIR / 'window_detection',
        max_files=None  # Process all files
    )
    
    # Save windows
    df_windows.to_parquet(windows_path)
    logger.info(f"Windows saved: {windows_path}")

# Display window statistics
logger.info("\n" + "="*70)
logger.info("WINDOW DETECTION SUMMARY")
logger.info("="*70)

for window in ['pre_arrival', 'p_wave', 's_wave', 'coda']:
    df_w = df_windows[df_windows['window_name'] == window]
    logger.info(f"\n{window.upper().replace('_', ' ')}:")
    logger.info(f"  Files with window: {df_w['file'].nunique()}")
    logger.info(f"  Duration range: [{df_w['duration_s'].min():.1f}, {df_w['duration_s'].max():.1f}] s")
    logger.info(f"  Mean duration: {df_w['duration_s'].mean():.1f} s")
    logger.info(f"  Samples range: [{df_w['n_samples'].min()}, {df_w['n_samples'].max()}]")

logger.info("\n" + "="*70)

## 4. Integration to velocity and displacement

Integrate acceleration once to obtain velocity, and twice to obtain displacement.
Baseline correction is crucial to prevent drift during integration.

In [ ]:
logger.info("Integrating acceleration to velocity...")
df_vel = integrate_to_velocity(df_acc, method='trapz')
check(len(df_vel) == len(df_acc), "Velocity computed for all samples")

logger.info("Integrating acceleration to displacement...")
df_disp = integrate_to_displacement(df_acc, method='trapz')
check(len(df_disp) == len(df_acc), "Displacement computed for all samples")

logger.info("Integration complete.")

## 5. Define analysis parameters

Set time lags (τ) and moment orders (q) for scaling analysis.

**Important**: τ values will be automatically filtered per window to fit the shortest window among all stations.

In [ ]:
# Tau values: logarithmically spaced from 1 to 10000 samples
# These will be filtered per window based on minimum window length
tau_values = np.unique(np.logspace(0, np.log10(10000), 100).astype(int))
logger.info(f"Tau values defined: {len(tau_values)} values from {tau_values.min()} to {tau_values.max()} samples")

# q values: moment orders from 0.5 to 5.0
q_values = np.arange(0.5, 5.1, 0.25)
logger.info(f"q values defined: {len(q_values)} values from {q_values.min()} to {q_values.max()}")

## 6. Windowed ensemble analysis

Compute increments, moments, and scaling exponents for all temporal windows and all three processes.

### Workflow:

1. **For each window** (pre-arrival, P-wave, S-wave, coda):
   - Find minimum window length across all stations
   - Filter τ values to fit shortest window
   - Compute increments with adaptive t₀ per station
   
2. **For each τ**:
   - Collect increments from all stations (ensemble)
   - Compute ensemble-averaged moments: $M_q(\tau) = \langle |\Delta x(\tau)|^q \rangle_{\text{stations}}$
   
3. **For each q**:
   - Fit scaling exponent: $M_q(\tau) \sim \tau^{\zeta(q)}$

This is done for acceleration, velocity, and displacement.

In [ ]:
logger.info("\n" + "#"*80)
logger.info("# STARTING WINDOWED ENSEMBLE ANALYSIS")
logger.info("#"*80)

results = compute_and_save_all_windowed(
    df_acc, df_vel, df_disp,
    df_windows,
    tau_values,
    q_values,
    output_dir=DATA_DIR
)

logger.info("\n" + "#"*80)
logger.info("# ANALYSIS COMPLETE")
logger.info("#"*80)

## 7. Results overview

Display scaling exponents ζ(q) for each window and process.

In [ ]:
logger.info("\n" + "="*80)
logger.info("SCALING EXPONENTS OVERVIEW")
logger.info("="*80)

for process in ['acceleration', 'velocity', 'displacement']:
    logger.info(f"\n{process.upper()}:")
    
    for window in ['pre_arrival', 'p_wave', 's_wave', 'coda']:
        df_zeta = results[process]['exponents'][window]
        
        if len(df_zeta) > 0:
            # Get ζ(2) and ζ(4) for comparison
            zeta_2 = df_zeta[df_zeta['q'] == 2.0]['zeta'].values
            zeta_4 = df_zeta[df_zeta['q'] == 4.0]['zeta'].values
            
            logger.info(f"  {window:12s}: "
                       f"ζ(2) = {zeta_2[0]:.3f}, "
                       f"ζ(4) = {zeta_4[0]:.3f}, "
                       f"n_points = {df_zeta['n_points'].iloc[0]}, "
                       f"R² = {df_zeta['r_squared'].mean():.3f}")
        else:
            logger.info(f"  {window:12s}: No data")

logger.info("\n" + "="*80)

## 8. Visualization

### 8.1 Displacement: ζ(q) across windows

Compare scaling exponents for the four temporal windows.

**Expected behavior**:
- Pre-arrival: ζ(q) ≈ 0 (no scaling, noise)
- P-wave: ζ(q) > 0, linear with slope a
- S-wave: ζ(q) > 0, linear with slope b ≠ a (different regime)
- Coda: ζ(q) ≈ 0 (return to noise)

In [ ]:
# Combine all displacement exponents
df_zeta_disp_all = pd.concat([
    results['displacement']['exponents']['pre_arrival'],
    results['displacement']['exponents']['p_wave'],
    results['displacement']['exponents']['s_wave'],
    results['displacement']['exponents']['coda']
], ignore_index=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

window_colors = {
    'pre_arrival': colors[0],
    'p_wave': colors[1],
    's_wave': colors[2],
    'coda': colors[3]
}

window_labels = {
    'pre_arrival': 'Pre-arrival',
    'p_wave': 'P-wave',
    's_wave': 'S-wave',
    'coda': 'Coda'
}

for window in ['pre_arrival', 'p_wave', 's_wave', 'coda']:
    df_w = df_zeta_disp_all[df_zeta_disp_all['window'] == window]
    if len(df_w) > 0:
        ax.plot(df_w['q'], df_w['zeta'], 'o-',
                color=window_colors[window],
                linewidth=2.5, markersize=8,
                label=window_labels[window])

# Reference: normal diffusion
q_ref = np.linspace(0.5, 5, 100)
ax.plot(q_ref, q_ref/2, '--', color='black', alpha=0.5,
        linewidth=1.5, label='Normal diffusion (ζ=q/2)')

ax.set_xlabel('q', fontsize=14)
ax.set_ylabel('ζ(q)', fontsize=14)
ax.set_title('Displacement: Scaling Exponents per Temporal Window', fontsize=16)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()

# Save
filepath = FIGURES_DIR / 'displacement' / 'zeta_comparison_windows.pdf'
plt.savefig(filepath, bbox_inches='tight')
logger.info(f"Saved: {filepath}")

plt.show()

### 8.2 Displacement: Moments M_q(τ) for P-wave

Verify power-law scaling: $M_q(\tau) \sim \tau^{\zeta(q)}$

In [ ]:
df_moments_p_disp = results['displacement']['moments']['p_wave']

# Plot moments for selected q values
fig, ax = plt.subplots(figsize=(10, 6))

q_to_plot = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]

for i, q in enumerate(q_to_plot):
    df_q = df_moments_p_disp[df_moments_p_disp['q'] == q]
    if len(df_q) > 0:
        ax.loglog(df_q['tau'], df_q['moment'], 'o-',
                 color=colors[i % len(colors)],
                 linewidth=2, markersize=6,
                 label=f'q = {q}')

ax.set_xlabel('τ (samples)', fontsize=14)
ax.set_ylabel('M_q(τ)', fontsize=14)
ax.set_title('P-wave: Moments vs Time Scale', fontsize=16)
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()

# Save
filepath = FIGURES_DIR / 'displacement' / 'moments_p_wave.pdf'
plt.savefig(filepath, bbox_inches='tight')
logger.info(f"Saved: {filepath}")

plt.show()

### 8.3 All processes: ζ(q) comparison for S-wave

Compare scaling behavior across acceleration, velocity, and displacement for S-wave window.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

process_labels = {
    'acceleration': 'Acceleration',
    'velocity': 'Velocity',
    'displacement': 'Displacement'
}

for i, process in enumerate(['acceleration', 'velocity', 'displacement']):
    df_zeta = results[process]['exponents']['s_wave']
    if len(df_zeta) > 0:
        ax.plot(df_zeta['q'], df_zeta['zeta'], 'o-',
                color=colors[i],
                linewidth=2.5, markersize=8,
                label=process_labels[process])

# Reference
q_ref = np.linspace(0.5, 5, 100)
ax.plot(q_ref, q_ref/2, '--', color='black', alpha=0.5,
        linewidth=1.5, label='Normal diffusion')

ax.set_xlabel('q', fontsize=14)
ax.set_ylabel('ζ(q)', fontsize=14)
ax.set_title('S-wave: Scaling Exponents for All Processes', fontsize=16)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()

# Save
filepath = FIGURES_DIR / 'zeta_comparison_processes_s_wave.pdf'
plt.savefig(filepath, bbox_inches='tight')
logger.info(f"Saved: {filepath}")

plt.show()

## 9. Summary

Create summary DataFrame with key results for each window and process.

In [ ]:
summary_rows = []

for process in ['acceleration', 'velocity', 'displacement']:
    for window in ['pre_arrival', 'p_wave', 's_wave', 'coda']:
        df_zeta = results[process]['exponents'][window]
        df_mom = results[process]['moments'][window]
        
        if len(df_zeta) > 0:
            # Extract key metrics
            zeta_1 = df_zeta[df_zeta['q'] == 1.0]['zeta'].values[0] if 1.0 in df_zeta['q'].values else np.nan
            zeta_2 = df_zeta[df_zeta['q'] == 2.0]['zeta'].values[0] if 2.0 in df_zeta['q'].values else np.nan
            zeta_4 = df_zeta[df_zeta['q'] == 4.0]['zeta'].values[0] if 4.0 in df_zeta['q'].values else np.nan
            
            # Ensemble size
            n_stations = df_mom['n_stations'].iloc[0] if len(df_mom) > 0 else 0
            
            # Fit quality
            mean_r2 = df_zeta['r_squared'].mean()
            
            summary_rows.append({
                'process': process,
                'window': window,
                'n_stations': n_stations,
                'zeta_q1': zeta_1,
                'zeta_q2': zeta_2,
                'zeta_q4': zeta_4,
                'mean_r2': mean_r2
            })

df_summary = pd.DataFrame(summary_rows)

logger.info("\nSUMMARY TABLE:")
display(df_summary)

# Save
df_summary.to_parquet(DATA_DIR / 'summary_windowed_ensemble.parquet', index=False)
logger.info(f"\nSummary saved: {DATA_DIR / 'summary_windowed_ensemble.parquet'}")